In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Kakapo.photometry import forced_photometry
from Kakapo.difference_image import create_diff_image, correct_motion_lightcurve
from Kakapo.kakapo import _check_tpf_type
from Kakapo.build_epsf import epsf_data_creation

from scipy.signal import fftconvolve
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import interp1d

from glob import glob
from tqdm import tqdm
import os

import lightkurve as lk

%matplotlib widget

In [ ]:

def _initial_set_up(directories, base_dir = "Data/"):
    for directory in directories:  # Iterate over each directory
        folder_path = os.path.join(os.getcwd(), base_dir + directory)  # Create the full path to the directory
        # print(folder_path)
        
        if os.path.exists(folder_path) and os.path.isdir(folder_path):  # Check if the folder exists
            for root, dirs, files in os.walk(folder_path, topdown=False):  # Walk through the directory tree
                for file in files: # Delete all files in the directory (in reverse order)
                    file_path = os.path.join(root, file)
                    try:
                        os.remove(file_path)
                    except Exception as e:
                        print(f"Failed to delete file {file_path}: {e}")
                
                for dir in dirs: # Delete all subdirectories
                    dir_path = os.path.join(root, dir)
                    try:
                        os.rmdir(dir_path)  # Remove the empty directory
                    except Exception as e:
                        print(f"Failed to delete directory {dir_path}: {e}")

            try: # Once all files and subdirectories are removed, remove the main directory itself
                os.rmdir(folder_path)  # Remove the now-empty directory
            except Exception as e:
                print(f"Failed to delete main directory {folder_path}: {e}")
        else:
            print(f"Directory does not exist: {folder_path}")
            
    for directory in directories:  # Iterate over each directory
        folder_path = os.path.join(os.getcwd(), base_dir + directory)  # Create the full path to the directory
        # print(folder_path)
        os.mkdir(folder_path)


def uno_reverse():
    # List of directories to clean
    directories = [
        "csv_files",
        "filtered_stars",
        "detected_events",
        "difference_arrays",
        "object_ids", 
        "figures",
        "temp_plots"
    ]
    
    _initial_set_up(directories, base_dir = "Data/")
    _initial_set_up([directories[0]], base_dir = "./")
    


In [ ]:
def _tpf_addition(tpf_info, tpf_input):
    
    if tpf_input.campaign is None:
        campaign = tpf_input.quarter
        mission = 'Kepler'
        print(f"Adding TPF {tpf_input.targetid} from {mission} quarter {campaign}")
    else:
        campaign = tpf_input.campaign
        mission = 'K2'
    
    tpf_info.loc[len(tpf_info)] = [mission, campaign, tpf_input.targetid, tpf_input.ra, tpf_input.dec, 
                                   tpf_input.flux.value, tpf_input.flux_err.value, tpf_input.quality, 
                                   tpf_input.pos_corr1, tpf_input.pos_corr2, tpf_input.time]
    
    return tpf_info
        
def _check_tpf_type(tpf_input):
    
    tpf_info = pd.DataFrame(columns=['mission', 'campaign', 'targetid', 'ra', 'dec', 'flux', 'flux_err', 
                                     'quality', 'pos_corr1', 'pos_corr2', 'time'])
    
    if isinstance(tpf_input, lk.targetpixelfile.KeplerTargetPixelFile):
        tpf_info = _tpf_addition(tpf_info, tpf_input)
        
    return tpf_info

In [ ]:
def download_tpfs():
    """
    """

    df = pd.read_csv('/Users/zgl12/testing.csv')#, encoding='ISO-8859-1')
    # df.to_csv('/Users/zgl12/testing.csv', encoding='utf-8', index=False)

    test_case = []

    for j in tqdm(range(7, len(df)), desc='Downloading TPFs'):
        test_case_df = df.iloc[j]

        camp = int(float(test_case_df['Field'].split('C')[-1]))
        test_case_df = df.iloc[j]
        
        if '/' in str(test_case_df['KIC/EPIC']):
            test_case_df['KIC/EPIC'] = float(test_case_df['KIC/EPIC'].split('/')[0])

        res = lk.search_targetpixelfile(test_case_df['KIC/EPIC'], mission='K2', campaign=camp)
        try:
            tpf0 = res[0].download(quality_bitmask="none")
            # tpf1 = res[1].download(quality_bitmask=0)
            test_case.append(tpf0)
        except:
            continue

    return test_case

def access_tpfs(stop = None):
    """
    """

    test_case = []

    lightkurve_file_folder = '/Users/zgl12/.lightkurve/cache/mastDownload/K2/'

    files = sorted(glob(lightkurve_file_folder + '*/*.fits.gz'))
    
    if stop is not None:
        files = files[:stop]

    for file in tqdm(files, desc='Reading TPFs'):
        tpf = lk.read(file, quality_bitmask = 'none')
        test_case.append(tpf)
        
    return test_case

In [ ]:
test_case = access_tpfs(stop = 6)
epsf_data = epsf_data_creation(test_case, path = '/Users/zgl12/Modules/Kakapo/', overwrite = False, stop_cond = 3000, sampling = 1)

plt.figure()
plt.imshow(epsf_data[2:-2, 2:-2], origin = 'lower')
plt.show()

In [ ]:
uno_reverse()
# start = time.time()

for i in range(5):
    tpf_info = _check_tpf_type(test_case[i])
    _, create_diff_image(tpf_info.iloc[0], plot=False, tol=0.001, mask_value=3e4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.stats import sigma_clipped_stats

%matplotlib widget

diff_file = '/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t206010203.npy'
diff_array = np.load(diff_file)

len(diff_array) // 50

plt.figure()
plt.imshow(diff_array[288])
plt.colorbar()
plt.show()

for i in range(10):
    data = diff_array[i*100]
    
    data = data.astype(np.float32)
    
    mean, median, stddev = sigma_clipped_stats(data, sigma = 3)
    
    plt.figure()
    plt.imshow(data)
    plt.colorbar()
    # plt.scatter(7.4, 7.6, color = 'r', marker = 'x')
    plt.scatter(7.94, 7.05, color = 'r', marker = 'x')
    
    plt.title(f"frame {i*100}, median {median:.2f}, std {stddev:.2f}")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.stats import sigma_clipped_stats

%matplotlib widget

diff_file = '/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy'
diff_array = np.load(diff_file)

len(diff_array) // 50

plt.figure()
plt.imshow(diff_array[288])
plt.colorbar()
plt.show()

for i in range(50):
    data = diff_array[i*30]
    
    data = data.astype(np.float32)
    
    mean, median, stddev = sigma_clipped_stats(data, sigma = 3)
    
    plt.figure()
    plt.imshow(data)
    plt.colorbar()
    # plt.scatter(7.4, 7.6, color = 'r', marker = 'x')
    plt.scatter(7.94, 7.05, color = 'r', marker = 'x')
    
    plt.title(f"frame {i*30}, median {median:.2f}, std {stddev:.2f}")
    plt.show()